In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
prod_desc = spark.read.format("parquet").load("abfss://bronze@intechaccstorage.dfs.core.windows.net/ProductDescription")
prod_desc.limit(5).display()

ProductDescriptionID,Description,rowguid,ModifiedDate,_rescued_data
3,Chromoly steel.,301eed3a-1a82-4855-99cb-2afe8290d641,2007-06-01 00:00:00.0000000,null
4,Aluminum alloy cups; large diameter spindle.,dfeba528-da11-4650-9d86-cafda7294eb0,2007-06-01 00:00:00.0000000,null
5,Aluminum alloy cups and a hollow axle.,f7178da7-1a7e-4997-8470-06737181305e,2007-06-01 00:00:00.0000000,null
8,"Suitable for any type of riding, on or off-road. Fits any budget. Smooth-shifting with a comfortable ride.",8e6746e5-ad97-46e2-bd24-fcea075c3b52,2007-06-01 00:00:00.0000000,null
64,"This bike delivers a high-level of performance on a budget. It is responsive and maneuverable, and offers peace-of-mind when you decide to go off-road.",7b1c4e90-85e2-4792-b47b-e0c424e2ec94,2007-06-01 00:00:00.0000000,null


In [0]:
prod_desc = prod_desc.withColumnRenamed('ProductDescriptionID', 'Product_description_id')\
                     .withColumnRenamed('ModifiedDate', 'Modified_date')\
                     .withColumnRenamed('Description', 'Product_description')


prod_desc = prod_desc.withColumn("Modified_date", to_date(col("Modified_date")))
                                 
prod_desc = prod_desc.drop("rowguid", "_rescued_data")   
prod_desc.columns     

['Product_description_id', 'Product_description', 'Modified_date']

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricksintech.silver.product_description
(
    Product_description_id STRING,
    Product_description STRING,
    Modified_date DATE
) 
USING DELTA
LOCATION 'abfss://silver@intechaccstorage.dfs.core.windows.net/product_description';

In [0]:
# Reference target Delta table
silver_table = DeltaTable.forName(spark, "databricksintech.silver.product_description")

# Execute the Merge (Upsert)
silver_table.alias("target").merge(
    source = prod_desc.alias("source"),
    condition = (
        "target.Product_description_id = source.Product_description_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
